# Christenson-Ximenez: Parse (sentence-level)

OHCO: `para_num, sent_num, token_num`

Source: existing `christenson-quiche-PARA.csv` (paragraph-level K'iche' text).

In [ ]:
import pandas as pd
import re

In [ ]:
src_id = 'christenson_ximenez'
para_path = '../../notebooks/christenson_ximenez/christenson-quiche-PARA.csv'

## Load paragraph-level text

In [ ]:
PARA = pd.read_csv(para_path)
PARA = PARA[PARA.doc_str.str.strip() != ''].copy()
print(f'{len(PARA)} paragraphs')
PARA.head()

## PARA to SENT — split on sentence-terminal punctuation

In [ ]:
SENT = (
    PARA
    .assign(doc_str=lambda df: df.doc_str.str.split(r'(?<=[.!?])\s+'))
    .explode('doc_str').dropna(subset=['doc_str'])
)
SENT = SENT[SENT.doc_str.str.strip() != ''].copy()
SENT['sent_num'] = SENT.groupby('para_num').cumcount()
SENT = SENT.reset_index(drop=True); SENT.index.name = 'doc_id'
DOC = SENT[['doc_str']]; DOCMAP = SENT[['para_num', 'sent_num']]
print(f'{len(DOC):,} sentences from {DOCMAP.para_num.nunique()} paragraphs')
DOCMAP.head()

## DOC to TOKEN

In [ ]:
TOKEN = DOC.doc_str.str.split(expand=True).stack().to_frame('token_str')
TOKEN.index.names = DOC.index.names + ['token_num']
TOKEN['term_str'] = TOKEN.token_str.str.lower().str.replace(r"[^a-z']", '', regex=True)
TOKEN = TOKEN[TOKEN.term_str != ''].dropna()
TOKEN

## Save

In [ ]:
TOKEN.to_csv(f'{src_id}-TOKEN.csv')
DOC.to_csv(f'{src_id}-DOC.csv')
DOCMAP.to_csv(f'{src_id}-DOCMAP.csv')
print('Saved to notebooks/doc_tables/')